## Bronze Layer to Silver Layer

In [ ]:
import polars as pl
from polars import col

import duckdb

### Ingest from Bronze

In [ ]:
con = duckdb.connect("../burger.db")

df_bronze = con.execute("SELECT * FROM bronze").pl()

#### Transformations

In [ ]:
# Filter out incorrect records
df_staging = df_bronze.filter((col('condition') == 'Graded') & (col('currency') == "USD"))

# Ensure Uniqueness
df_staging = df_staging.unique(['item_id', 'loaded_at'])

# Title to uppercase for consistency
df_staging = df_staging.with_columns(
    col('title').str.to_uppercase()
)

# Ensure PSA 10 in title
df_staging = df_staging.filter((col('title').str.contains('PSA')) & (col('title').str.contains('10')))

# Assign Daily Relative Value (Good Price, Fair Price, Overpriced)



display(df_staging)


#### Load to Silver Layer

In [ ]:
df_staging.write_csv('example.csv')

In [ ]:
con.close()